# Lecturer voice — full cloud pipeline (F5-TTS, MIT)

Everything runs **on Kaggle**, nothing local. End-to-end:

1. Download `lecture.wav` from **Google Drive** → 24 kHz mono
2. **Transcribe + diarize** → keep the lecturer's clean clips
3. Build a **Parquet** dataset and **push it to HuggingFace**
4. **Download** the base Russian F5-TTS model from HuggingFace
5. **Fine-tune**
6. **Upload** the fine-tuned model to HuggingFace

F5-TTS is MIT licensed (unlike XTTS v2 / CPML). We fine-tune from a Russian
checkpoint so Cyrillic is already in the vocab — no vocab-extension surgery.

**Before running:** Settings → Accelerator → **GPU**; Add-ons → Secrets → add
`HF_TOKEN` (a *write* token). Accept the pyannote license at
huggingface.co/pyannote/speaker-diarization-3.1 with that same account.
Then edit the **CONFIG** cell.

## 0 · Install + config

In [ ]:
# Clone + editable install so src/, data/ and ckpts/ all live under F5_ROOT
# (a plain pip install has no src/ tree and f5_tts.__file__ is None).
F5_ROOT = '/kaggle/working/F5-TTS'
!git clone -q https://github.com/SWivid/F5-TTS.git {F5_ROOT}
!pip -q install -e {F5_ROOT}
!pip -q install openai-whisper pyannote.audio soundfile librosa \
    datasets huggingface_hub gdown
# ffmpeg is preinstalled on Kaggle.
import os
print('F5-TTS at', F5_ROOT)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# ============================== CONFIG ==============================
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')   # write token
os.environ['HF_TOKEN'] = HF_TOKEN

# --- source audio: data/lecture.wav on Google Drive ---
# https://drive.google.com/file/d/<GDRIVE_ID>/view
GDRIVE_ID = '1wOkn2580v2dpvCRLjT03HLy1iaWlCpws'

# --- HuggingFace repos (your handle) ---
HF_USER          = 'cyttic'
HF_DATASET_REPO  = f'{HF_USER}/lecturer-ru-dataset'    # parquet goes here
HF_MODEL_REPO    = f'{HF_USER}/lecturer-ru-f5tts'      # fine-tuned model goes here

# --- base Russian checkpoint to fine-tune FROM ---
PRETRAIN_REPO  = 'Misha24-10/F5-TTS_RUSSIAN'
PRETRAIN_CKPT  = 'F5TTS_v1_Base_v2/model_last_inference.safetensors'
PRETRAIN_VOCAB = 'F5TTS_v1_Base/vocab.txt'
EXP_NAME       = 'F5TTS_v1_Base'   # MUST match the checkpoint architecture

# --- prep params ---
LANGUAGE      = 'ru'
WHISPER_MODEL = 'large-v3'
SR            = 24000              # F5-TTS sample rate
MIN_DUR, MAX_DUR = 3.0, 15.0
MIN_LOGPROB, MAX_NOSPEECH = -0.5, 0.3
DATASET_NAME  = 'lecturer_ru'     # F5-TTS data/<name>_custom/
# ===================================================================

## 1 · Download `lecture.wav` from Google Drive → 24 kHz mono

In [ ]:
AUDIO = '/kaggle/working/lecture.wav'
if not GDRIVE_ID:
    raise SystemExit('Set GDRIVE_ID in CONFIG (the id of data/lecture.wav on Drive).')

!gdown {GDRIVE_ID} -O /kaggle/working/lecture_src.wav
# Resample to what F5-TTS expects (24 kHz mono), whatever the source rate is.
!ffmpeg -y -i /kaggle/working/lecture_src.wav -ar {SR} -ac 1 {AUDIO}
!rm /kaggle/working/lecture_src.wav

import soundfile as sf
info = sf.info(AUDIO)
print(f'{AUDIO}: {info.duration/60:.1f} min, {info.samplerate} Hz, {info.channels} ch')

## 2 · Transcribe + diarize → lecturer's clean clips

Audio is loaded via `soundfile` and passed to pyannote in-memory (bypasses the
often-broken torchcodec loader). Clips are cut at pyannote speaker-turn
boundaries, so each is single-speaker; we keep the speaker with the most total
speech (the lecturer).

In [ ]:
import torch, whisper

# --- transcribe ---
print('Transcribing...')
wmodel = whisper.load_model(WHISPER_MODEL, device='cuda')
result = wmodel.transcribe(AUDIO, language=LANGUAGE, verbose=False)
wsegs = [{'start': s['start'], 'end': s['end'], 'text': s['text'].strip(),
          'avg_logprob': s.get('avg_logprob', 0.0),
          'no_speech_prob': s.get('no_speech_prob', 0.0)}
         for s in result['segments'] if s['text'].strip()]
print(f'  {len(wsegs)} whisper segments')
del wmodel; torch.cuda.empty_cache()

In [ ]:
# --- diarize ---
from pyannote.audio import Pipeline
print('Diarizing...')
try:    # pyannote.audio >= 4.0
    dia = Pipeline.from_pretrained('pyannote/speaker-diarization-3.1', token=HF_TOKEN)
except TypeError:   # pyannote.audio 3.x
    dia = Pipeline.from_pretrained('pyannote/speaker-diarization-3.1', use_auth_token=HF_TOKEN)
dia.to(torch.device('cuda'))

wav, srate = sf.read(AUDIO, dtype='float32', always_2d=True)
out = dia({'waveform': torch.from_numpy(wav.T), 'sample_rate': srate})
ann = getattr(out, 'exclusive_speaker_diarization',
              getattr(out, 'speaker_diarization', out))

turns = sorted(
    ({'start': t.start, 'end': t.end, 'speaker': s}
     for t, _, s in ann.itertracks(yield_label=True)),
    key=lambda x: x['start'])
totals = {}
for t in turns:
    totals[t['speaker']] = totals.get(t['speaker'], 0.0) + (t['end'] - t['start'])
lecturer = max(totals, key=totals.get)
print('speakers (sec):', {k: round(v, 1) for k, v in totals.items()})
print('lecturer:', lecturer)

In [ ]:
# --- match whisper text to lecturer turns, quality filter ---
clips = []
for tn in turns:
    if tn['speaker'] != lecturer:
        continue
    dur = tn['end'] - tn['start']
    if not (MIN_DUR <= dur <= MAX_DUR):
        continue
    overlap = [s for s in wsegs
               if min(s['end'], tn['end']) - max(s['start'], tn['start']) > 0
               and s['avg_logprob'] >= MIN_LOGPROB
               and s['no_speech_prob'] <= MAX_NOSPEECH]
    text = ' '.join(s['text'] for s in overlap).strip()
    if text:
        clips.append({'start': tn['start'], 'end': tn['end'], 'text': text})
print(f'{len(clips)} clean lecturer clips '
      f'({sum(c["end"]-c["start"] for c in clips)/60:.1f} min)')

## 3 · Build Parquet → push to HuggingFace dataset

In [ ]:
import io, librosa, numpy as np
from datasets import Dataset, Audio

full, fsr = sf.read(AUDIO, dtype='float32', always_2d=True)
full = full.mean(axis=1)   # mono

rows = {'audio': [], 'text': [], 'duration': []}
for i, c in enumerate(clips, 1):
    seg = full[int(c['start']*fsr):int(c['end']*fsr)]
    if fsr != SR:
        seg = librosa.resample(seg, orig_sr=fsr, target_sr=SR)
    buf = io.BytesIO()
    sf.write(buf, seg, SR, format='WAV', subtype='PCM_16')
    rows['audio'].append({'bytes': buf.getvalue(), 'path': f'clip_{i:04d}.wav'})
    rows['text'].append(c['text'])
    rows['duration'].append(round(c['end']-c['start'], 3))

ds = Dataset.from_dict(rows).cast_column('audio', Audio(sampling_rate=SR))
ds.push_to_hub(HF_DATASET_REPO, token=HF_TOKEN, private=True)
print('pushed dataset ->', f'https://huggingface.co/datasets/{HF_DATASET_REPO}')

## 3b · (restart only) Pull the prepared dataset back from HF

If the session died after the dataset was pushed, skip sections 1–3 entirely:
run cells 0 (install) and CONFIG, then this, then continue from section 4.

In [ ]:
# Rebuild `rows` from the HF dataset — same shape section 3 produces, so the
# prepare cell in section 5 works unchanged. Audio stays as raw wav bytes
# (decode=False) so no torchcodec/decoder is ever involved.
from datasets import load_dataset, Audio
ds_hf = load_dataset(HF_DATASET_REPO, split='train', token=HF_TOKEN)
ds_hf = ds_hf.cast_column('audio', Audio(decode=False))
rows = {'audio': [{'bytes': r['audio']['bytes']} for r in ds_hf],
        'text': ds_hf['text'], 'duration': ds_hf['duration']}
print(len(rows['text']), 'clips,', round(sum(rows['duration'])/60, 1), 'min')

## 4 · Download base model from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download
PRE = '/kaggle/working/pretrained'
ckpt  = hf_hub_download(PRETRAIN_REPO, PRETRAIN_CKPT, local_dir=PRE)
vocab = hf_hub_download(PRETRAIN_REPO, PRETRAIN_VOCAB, local_dir=PRE)
print('ckpt :', ckpt)
print('vocab:', vocab)

## 5 · Prepare F5-TTS data + fine-tune

In [ ]:
# Explode the in-memory dataset to wavs/ + metadata.csv (header 'audio_file|text',
# ABSOLUTE paths). Decode the stored bytes with soundfile (not the Audio decoder).
import csv, shutil
WORK, WAVS = '/kaggle/working/ds', '/kaggle/working/ds/wavs'
os.makedirs(WAVS, exist_ok=True)
meta = f'{WORK}/metadata.csv'
with open(meta, 'w', encoding='utf-8', newline='') as f:
    w = csv.writer(f, delimiter='|'); w.writerow(['audio_file', 'text'])
    for i, r in enumerate(rows['audio'], 1):
        p = f'{WAVS}/clip_{i:04d}.wav'
        d, s = sf.read(io.BytesIO(r['bytes']), dtype='float32'); sf.write(p, d, s)
        w.writerow([p, rows['text'][i-1]])

# Train runs with --tokenizer custom, and finetune_cli loads the dataset from
# data/<DATASET_NAME>_<tokenizer> — so the dir suffix must be _custom.
OUT = f'{F5_ROOT}/data/{DATASET_NAME}_custom'
os.makedirs(OUT, exist_ok=True)
!python {F5_ROOT}/src/f5_tts/train/datasets/prepare_csv_wavs.py {meta} {OUT}
shutil.copy(vocab, f'{OUT}/vocab.txt')   # align token ids with the checkpoint
print('prepared:', os.listdir(OUT))

In [ ]:
# Guardrails (learned the hard way): check disk headroom (~10 GB needed; a full
# /kaggle/working hangs mid-epoch with the GPU idle), cap dataloader workers at
# Kaggle's 4 CPUs (16 can deadlock), keep only 1 rolling checkpoint.
!df -h /kaggle/working | tail -1
!sed -i 's/num_workers=16/num_workers=4/' \
    {F5_ROOT}/src/f5_tts/train/finetune_cli.py {F5_ROOT}/src/f5_tts/model/trainer.py

!accelerate launch --num_processes 1 {F5_ROOT}/src/f5_tts/train/finetune_cli.py \
  --exp_name {EXP_NAME} --dataset_name {DATASET_NAME} --finetune \
  --pretrain {ckpt} --tokenizer custom --tokenizer_path {vocab} \
  --learning_rate 1e-5 --batch_size_per_gpu 2000 --batch_size_type frame \
  --max_samples 64 --grad_accumulation_steps 2 --max_grad_norm 1.0 \
  --epochs 45 --num_warmup_updates 200 \
  --save_per_updates 1000 --keep_last_n_checkpoints 1 \
  --last_per_updates 500 --log_samples
# Auto-resumes from ckpts/<dataset>/model_last.pt if present (saved every 500 updates).
# OOM on T4? lower --batch_size_per_gpu (e.g. 1200), raise --grad_accumulation_steps.
# state_dict mismatch on load? EXP_NAME doesn't match the ckpt: try F5TTS_Base.

## 6 · Upload fine-tuned model to HuggingFace

In [ ]:
import glob
from huggingface_hub import HfApi
CKPT_DIR = f'{F5_ROOT}/ckpts/{DATASET_NAME}'
final = sorted(glob.glob(f'{CKPT_DIR}/model_*.pt'))[-1]
print('uploading', final)

api = HfApi(token=HF_TOKEN)
api.create_repo(HF_MODEL_REPO, repo_type='model', private=True, exist_ok=True)
api.upload_file(path_or_fileobj=final, path_in_repo='model_last.pt',
                repo_id=HF_MODEL_REPO)
api.upload_file(path_or_fileobj=vocab, path_in_repo='vocab.txt',
                repo_id=HF_MODEL_REPO)
print('done ->', f'https://huggingface.co/{HF_MODEL_REPO}')

## (optional) Quick listen before you trust it

In [ ]:
ref_wav = f'{WAVS}/clip_0001.wav'
ref_txt = open(meta, encoding='utf-8').read().splitlines()[1].split('|', 1)[1]
!f5-tts_infer-cli --model {EXP_NAME} --ckpt_file {final} --vocab_file {vocab} \
  --ref_audio {ref_wav} --ref_text "{ref_txt}" \
  --gen_text "Привет! Это тест клонированного голоса лектора." \
  --output_dir /kaggle/working/test_out
from IPython.display import Audio
Audio(sorted(glob.glob('/kaggle/working/test_out/*.wav'))[-1])